In [21]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
import pandas as pd
import os
import random

from tqdm import tqdm 
svanoo_myanimelist_dataset_path = kagglehub.dataset_download('svanoo/myanimelist-dataset')

print('Data source import complete.')


Data source import complete.


In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(f'{svanoo_myanimelist_dataset_path}'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\anime.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\anime_anime.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000000.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000001.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000002.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000003.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000004.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000000000005.csv
C:\Users\Heart\.cache\kagglehub\datasets\svanoo\myanimelist-dataset\versions\2\user_anime000

In [23]:
NUM_USER_ANIME_FILES_TO_LOAD = 70
NUM_USER_ANIME_FILES_TO_LOAD = min(NUM_USER_ANIME_FILES_TO_LOAD, 70)

In [37]:
# Set random seed for reproducibility
random.seed(42)
output_file = "filtered_user_anime.csv"
columns_to_keep = ['anime_id', 'user_id', 'score', 'favorite']


# Parameter to control what percentage of reviews to keep
sample_percentage = 1  # Keep % of reviews (adjust as needed)

if os.path.exists(output_file):
    os.remove(output_file)

first = True
total_rows_kept = 0

for file_path in tqdm(list_files, desc="Filtering user_anime files"):
    # Read in chunks to handle memory better if needed
    df = pd.read_csv(
        file_path,
        sep='\t',
        keep_default_na=False,
        na_values=("-------",""),
        low_memory=False,
        usecols=['anime_id', 'user_id', 'score', 'favorite']  # Only read needed columns
    ).dropna()
    df = df[df["anime_id"].isin(top_animes)]
    means = user.loc[df["user_id"],"mean_score"].values
    liked =  (means>= df["score"].values) | df["favorite"].values
    df = df[liked == 1]
    df.drop(["score","favorite"],axis=1,inplace=True)
    df.dropna(inplace=True)
    if not df.empty:
        original_count = len(df)

        # More aggressive sampling for large datasets
        if len(df) > 10:  # Only sample if we have enough rows
            n_samples = max(1, int(len(df) * sample_percentage))
            df = df.sample(n=n_samples, random_state=42)
        df = df.sort_values("anime_id")
        df.to_csv(output_file, mode='a', index=False, header=first)
        total_rows_kept += len(df)
        df_new=pd.read_csv(output_file)
        if df.isna().any().any():
            print(df_new[df_new.isna().any(axis="columns")])
            print(df.index,df_new.index)
            raise IndexError

        first = False

print(f"\nTotal rows kept across all files: {total_rows_kept:,}")

Filtering user_anime files: 100%|██████████| 70/70 [20:10<00:00, 17.29s/it]


Total rows kept across all files: 50,922,994
